# CTD Feature Selection Pipeline

Two-stage selection on **`ctd__*`** columns from enriched DTI datasets in `data/processed/`.

1. **Stage 1 — Variance threshold:** drop features with $\sigma^2 \le 0.01$.
2. **Stage 2 — Collinearity filter:** for pairs with $|r| > 0.85$, drop the lower-variance feature.

Outputs are written to **`data/trainready/`** (does not modify `data/processed/`).

In [ ]:
%pip install -q pandas numpy pyarrow scikit-learn


In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)


In [ ]:
# --- Paths & hyperparameters ---
ROOT = Path.cwd()
INPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "data" / "trainready"
REPORT_DIR = OUTPUT_DIR / "reports"

DATASETS = ["davis", "kiba", "bindingdb_kd"]

VARIANCE_THRESH = 0.01   # Stage 1: keep variance > 0.01
CORR_THRESH = 0.85       # Stage 2: |Pearson r| threshold

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)


In [ ]:
def select_significant_ctd_features(
    df: pd.DataFrame,
    variance_thresh: float = 0.01,
    corr_thresh: float = 0.85,
) -> Tuple[pd.DataFrame, List[str], Dict[str, Any]]:
    """Filter CTD columns; preserve all non-CTD columns and row index.

    Returns
    -------
    final_df, kept_ctd_cols, report_dict
    """
    original_index = df.index
    print(f"Initial dataframe shape: {df.shape}")

    ctd_cols = [c for c in df.columns if c.startswith("ctd__")]
    if not ctd_cols:
        raise ValueError("No columns starting with 'ctd__' found.")

    df_ctd = df[ctd_cols].copy()
    df_ctd = df_ctd.apply(pd.to_numeric, errors="coerce")
    # Median imputation for selection only (rows with missing UniProt / CTD)
    df_ctd = df_ctd.fillna(df_ctd.median(numeric_only=True))
    print(f"Identified {len(ctd_cols)} raw CTD features.")

    # ----- Stage 1: Variance threshold -----
    selector = VarianceThreshold(threshold=variance_thresh)
    selector.fit(df_ctd)
    low_var_mask = selector.get_support()
    passed_var_cols = [c for c, keep in zip(ctd_cols, low_var_mask) if keep]
    dropped_stage1 = [c for c in ctd_cols if c not in passed_var_cols]
    df_stage1 = df_ctd[passed_var_cols]
    print(
        f"Stage 1: {len(passed_var_cols)} kept, {len(dropped_stage1)} dropped "
        f"(variance <= {variance_thresh})."
    )

    # ----- Stage 2: Absolute correlation filter -----
    feat_variances = df_stage1.var()
    corr_matrix = df_stage1.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    cols_to_drop = set()
    for col in upper.columns:
        correlated_with = upper.index[upper[col] > corr_thresh].tolist()
        if not correlated_with:
            continue
        current_var = feat_variances[col]
        for rival in correlated_with:
            if current_var < feat_variances[rival]:
                cols_to_drop.add(col)
                break
            cols_to_drop.add(rival)

    passed_corr_cols = [c for c in passed_var_cols if c not in cols_to_drop]
    dropped_stage2 = sorted(cols_to_drop)
    print(f"Stage 2: dropped {len(dropped_stage2)} correlated features (|r| > {corr_thresh}).")
    print(f"Final CTD features: {len(passed_corr_cols)}")

    non_ctd_cols = [c for c in df.columns if not c.startswith("ctd__")]
    final_df = pd.concat([df[non_ctd_cols], df[passed_corr_cols]], axis=1)
    final_df.index = original_index
    print(f"Final shape: {final_df.shape}\n")

    report = {
        "variance_threshold": variance_thresh,
        "correlation_threshold": corr_thresh,
        "initial_ctd_count": len(ctd_cols),
        "kept_ctd_count": len(passed_corr_cols),
        "dropped_stage1_low_variance": dropped_stage1,
        "dropped_stage2_high_correlation": dropped_stage2,
        "kept_ctd_features": passed_corr_cols,
    }
    return final_df, passed_corr_cols, report


def drop_paac_fallback_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """Remove non-standard PAAC columns (keep only propy ``paac_APAAC*``)."""
    drop_cols = [
        c for c in df.columns
        if c.startswith("paac_") and not c.startswith("paac_APAAC")
    ]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df, drop_cols


def align_to_canonical_columns(df: pd.DataFrame, canonical_columns: List[str]) -> pd.DataFrame:
    """Reindex to the shared train-ready schema (adds NaN for missing features)."""
    return df.reindex(columns=canonical_columns)


## Process each dataset

Load from `data/processed/`, apply selection, save to `data/trainready/`.

In [ ]:
all_reports: Dict[str, Any] = {}
summaries: List[Dict[str, Any]] = []
outputs: Dict[str, pd.DataFrame] = {}

for name in DATASETS:
    in_path = INPUT_DIR / f"{name}_enriched.parquet"
    if not in_path.exists():
        raise FileNotFoundError(f"Missing input: {in_path}")

    print("=" * 60)
    print(f"DATASET: {name}")
    print("=" * 60)

    df = pd.read_parquet(in_path)
    assert df.index.is_unique, "Row index must be unique for alignment."

    out_df, kept_cols, report = select_significant_ctd_features(
        df,
        variance_thresh=VARIANCE_THRESH,
        corr_thresh=CORR_THRESH,
    )

    # BindingDB: remove propy PAAC fallback columns (paac_A, paac_dipep_*, etc.)
    if name == "bindingdb_kd":
        out_df, dropped_paac = drop_paac_fallback_columns(out_df)
        report["dropped_paac_fallback"] = dropped_paac
        print(f"Removed {len(dropped_paac)} non-standard PAAC fallback columns.")

    report["dataset"] = name
    report["input_rows"] = int(len(df))
    report["input_columns"] = int(len(df.columns))
    report["columns_after_ctd_and_paac_cleanup"] = int(len(out_df.columns))
    outputs[name] = out_df
    all_reports[name] = report

# Uniform schema: intersection of columns across all datasets (after PAAC cleanup)
canonical_columns = list(outputs["davis"].columns)
for name in DATASETS[1:]:
    canonical_columns = [c for c in canonical_columns if c in outputs[name].columns]
for extra_name in DATASETS[1:]:
    for c in outputs[extra_name].columns:
        if c not in canonical_columns and c.startswith("ctd__"):
            pass  # CTD extras not in intersection are dropped
print(f"\nCanonical train-ready columns: {len(canonical_columns)}")

for name in DATASETS:
    aligned = align_to_canonical_columns(outputs[name], canonical_columns)
    report = all_reports[name]
    report["canonical_columns"] = canonical_columns
    report["output_columns"] = int(len(aligned.columns))
    report["output_rows"] = int(len(aligned))

    out_path = OUTPUT_DIR / f"{name}_trainready.parquet"
    aligned.to_parquet(out_path, index=False)

    report_path = REPORT_DIR / f"dropped_ctd_features_{name}.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

    summaries.append({
        "dataset": name,
        "rows": len(aligned),
        "cols_in": report["input_columns"],
        "cols_after_ctd": report["columns_after_ctd_and_paac_cleanup"],
        "cols_out": len(aligned.columns),
        "ctd_in": report["initial_ctd_count"],
        "ctd_out": report["kept_ctd_count"],
        "output": str(out_path),
    })
    print(f"{name}: saved {aligned.shape} (canonical schema)")

combined_path = REPORT_DIR / "dropped_ctd_features.json"
combined_path.write_text(json.dumps(all_reports, indent=2), encoding="utf-8")
print("Wrote combined report:", combined_path)


In [ ]:
summary_df = pd.DataFrame(summaries)
display(summary_df)


## Validation

Confirm row counts match inputs and `data/processed/` files were not modified.

In [ ]:
for name in DATASETS:
    src = pd.read_parquet(INPUT_DIR / f"{name}_enriched.parquet")
    dst = pd.read_parquet(OUTPUT_DIR / f"{name}_trainready.parquet")

    assert len(src) == len(dst), f"{name}: row count mismatch"
    for col in ["Drug_ID", "Target_ID", "drug_smiles", "uniprot_id"]:
        if col in src.columns:
            assert src[col].equals(dst[col]), f"{name}: misaligned column {col}"

    ctd_remaining = [c for c in dst.columns if c.startswith("ctd__")]
    paac_cols = [c for c in dst.columns if c.startswith("paac_")]
    bad_paac = [c for c in paac_cols if not c.startswith("paac_APAAC")]
    assert not bad_paac, f"{name}: non-standard PAAC columns remain: {bad_paac[:5]}"
    print(
        f"{name}: OK — {len(src)} rows, CTD {len(ctd_remaining)}, "
        f"PAAC {len(paac_cols)} (APAAC only), cols={len(dst.columns)}"
    )

# Cross-dataset column uniformity
shapes = {n: set(pd.read_parquet(OUTPUT_DIR / f"{n}_trainready.parquet").columns) for n in DATASETS}
assert shapes["davis"] == shapes["kiba"] == shapes["bindingdb_kd"], "Column mismatch across datasets"
print(f"\nUniform schema verified: {len(shapes['davis'])} columns on all datasets")
print("Processed inputs untouched:", all(p.exists() for p in INPUT_DIR.glob("*_enriched.parquet")))
